In [1]:
INPUT_DATA : str = "MAN" # MAN, WOMAN, ADULT, CHILDREN
TARGET_DATA : str = "WOMAN" # MAN, WOMAN, ADULT, CHILDREN
EPSILON : float = 1.0

# Imports

In [2]:
import os, sys
sys.path.append("..")
sys.path.append("../ALAE")

import comet_ml
import torch
import torch.nn as nn
import numpy as np

from src.distributions import TensorSampler
from tqdm import tqdm

# import wandb
from matplotlib import pyplot as plt
import torch.distributions as TD

from alae_ffhq_inference import load_model, encode, decode

DEVICE = 'cuda:0'

## Basic Config

In [3]:
DIM = 512
assert DIM > 1

OUTPUT_SEED = 0xBADBEEF
MAX_STEPS = 10000
CONTINUE = -1

In [4]:
EXP_NAME = f'EgNOT_ALAE_{INPUT_DATA}_TO_{TARGET_DATA}_EPSILON_{EPSILON}'
OUTPUT_PATH = '../checkpoints/{}'.format(EXP_NAME)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

# Data loading

### TO DOWNLOAD PRE-PROCESSED ALAE DATA, UNCOMMENT THE CODE OF THE NEXT CELL.

In [5]:
# import gdown
# import os

# urls = {
#     "../data/age.npy": "https://drive.google.com/uc?id=1Vi6NzxCsS23GBNq48E-97Z9UuIuNaxPJ",
#     "../data/gender.npy": "https://drive.google.com/uc?id=1SEdsmQGL3mOok1CPTBEfc_O1750fGRtf",
#     "../data/latents.npy": "https://drive.google.com/uc?id=1ENhiTRsHtSjIjoRu1xYprcpNd8M9aVu8",
#     "../data/test_images.npy": "https://drive.google.com/uc?id=1SjBWWlPjq-dxX4kxzW-Zn3iUR3po8Z0i",
# }

# for name, url in urls.items():
#     gdown.download(url, os.path.join(f"{name}"), quiet=False)

In [6]:
# To download data use

train_size = 60000
test_size = 10000

latents = np.load("../data/latents.npy")
gender = np.load("../data/gender.npy")
age = np.load("../data/age.npy")
test_inp_images = np.load("../data/test_images.npy")

train_latents, test_latents = latents[:train_size], latents[train_size:]
train_gender, test_gender = gender[:train_size], gender[train_size:]
train_age, test_age = age[:train_size], age[train_size:]

if INPUT_DATA == "MAN":
    x_inds_train = np.arange(train_size)[(train_gender == "male").reshape(-1)]
    x_inds_test = np.arange(test_size)[(test_gender == "male").reshape(-1)]
elif INPUT_DATA == "WOMAN":
    x_inds_train = np.arange(train_size)[(train_gender == "female").reshape(-1)]
    x_inds_test = np.arange(test_size)[(test_gender == "female").reshape(-1)]
elif INPUT_DATA == "ADULT":
    x_inds_train = np.arange(train_size)[
        (train_age >= 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    x_inds_test = np.arange(test_size)[
        (test_age >= 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
elif INPUT_DATA == "CHILDREN":
    x_inds_train = np.arange(train_size)[
        (train_age < 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    x_inds_test = np.arange(test_size)[
        (test_age < 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
x_data_train = train_latents[x_inds_train]
x_data_test = test_latents[x_inds_test]

if TARGET_DATA == "MAN":
    y_inds_train = np.arange(train_size)[(train_gender == "male").reshape(-1)]
    y_inds_test = np.arange(test_size)[(test_gender == "male").reshape(-1)]
elif TARGET_DATA == "WOMAN":
    y_inds_train = np.arange(train_size)[(train_gender == "female").reshape(-1)]
    y_inds_test = np.arange(test_size)[(test_gender == "female").reshape(-1)]
elif TARGET_DATA == "ADULT":
    y_inds_train = np.arange(train_size)[
        (train_age >= 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    y_inds_test = np.arange(test_size)[
        (test_age >= 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
elif TARGET_DATA == "CHILDREN":
    y_inds_train = np.arange(train_size)[
        (train_age < 18).reshape(-1)*(train_age != -1).reshape(-1)
    ]
    y_inds_test = np.arange(test_size)[
        (test_age < 18).reshape(-1)*(test_age != -1).reshape(-1)
    ]
y_data_train = train_latents[y_inds_train]
y_data_test = test_latents[y_inds_test]

X_train = torch.tensor(x_data_train)
Y_train = torch.tensor(y_data_train)

X_test = torch.tensor(x_data_test)
Y_test = torch.tensor(y_data_test)

X_sampler = TensorSampler(X_train, device=DEVICE)
Y_sampler = TensorSampler(Y_train, device=DEVICE)

# Model initialisation

## EgNOT

In [7]:
# imports
from src.utils import Config
from src.models import FullyConnectedMLPwithConfig

# EOT
from src.egeot import EgEOT_l2sq_Mixin
from src.egeot import SampleBufferGeneric, SampleBufferStatic

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# PARAMETERS
CONFIG = Config()

BATCH_SIZE = 128

CONFIG.BATCH_SIZE = BATCH_SIZE
CONFIG.DEVICE = DEVICE
CONFIG.RESOLUTION = DIM # 512 - latent ALAE dim

CONFIG.HREG = EPSILON
CONFIG.ALPHA = 0.00 # loss += alpha * (pos_out ** 2 + neg_out ** 2)

# langevin dynamics parameters
CONFIG.ENERGY_SAMPLING_METHOD = 'langevin_classic' #'classic', 'pseudo'
CONFIG.LANGEVIN_THRESH = None
CONFIG.LANGEVIN_SAMPLING_NOISE = 0.05
CONFIG.ENERGY_SAMPLING_ITERATIONS = 100
CONFIG.TEST_ENERGY_SAMPLING_ITERATIONS = 1000 # CONFIG.ENERGY_SAMPLING_ITERATIONS * 50
CONFIG.LANGEVIN_DECAY = 1.0
CONFIG.LANGEVIN_SCORE_COEFFICIENT = 0.1
CONFIG.LANGEVIN_COST_COEFFICIENT = CONFIG.LANGEVIN_SAMPLING_NOISE ** 2
CONFIG.ENERGY_SAMPLING_STEP = 1.0
CONFIG.REFERENCE_DATA_NOISE_SIGMA = 0.00
CONFIG.ENERGY_SAMPLING_NO_PROJECT_DATA = True

#-----------------------------------------------
# nonactual for langevin_classic sampling method
# CONFIG.PSEUDO_LANGEVIN_GRAD_PROJ_TYPE = 'none' # 'value', 'norm', 'none'
# CONFIG.PSEUDO_LANGEVIN_NORM_THRESH = 1.
# CONFIG.PSEUDO_LANGEVIN_VALUE_THRESH = 0.01
# CONFIG.PSEUDO_LANGEVIN_NOISE = 0.005
#---------------------------------------------

# training
CONFIG.LR = 1e-4
CONFIG.ADAM_BETAS = (0.2, 0.99)
CONFIG.CLIP_GRAD_NORM = 1.

Sample Buffer

In [9]:
y_data_mean = torch.tensor(y_data_train.mean(axis=0)).to(dtype=torch.float32, device=DEVICE)
y_data_var = torch.tensor(y_data_train.std(axis=0)).to(dtype=torch.float32, device=DEVICE) 
var_multiplier = 3.
init_noise_distribution = TD.Independent(TD.Normal(y_data_mean, var_multiplier * y_data_var), 1)
init_noise_gen = lambda shape: init_noise_distribution.sample(shape)

sample_buffer = SampleBufferStatic(
    init_noise_gen, 
    torch.tensor(x_data_train).to(dtype=torch.float32), 
    device = DEVICE,
)

EgEOT model + optimizer

In [10]:
class EgEOT_l2sq(EgEOT_l2sq_Mixin, FullyConnectedMLPwithConfig):

    def __init__(
        self, 
        sample_buffer, 
        config, 
        hiddens=[1024, 512, 256],
        activation = lambda : nn.ReLU()
    ):
        super().__init__(sample_buffer, config, CONFIG.RESOLUTION, hiddens, 1, activation_gen=activation)

egeot_model = EgEOT_l2sq(sample_buffer, CONFIG).to(DEVICE)
optimizer = torch.optim.Adam(egeot_model.potential.parameters(), lr = CONFIG.LR, betas = CONFIG.ADAM_BETAS)

In [11]:
# torch.manual_seed(OUTPUT_SEED); np.random.seed(OUTPUT_SEED)

# D = LightSB(dim=DIM, n_potentials=N_POTENTIALS, epsilon=EPSILON,
#             sampling_batch_size=SAMPLING_BATCH_SIZE, S_diagonal_init=0.1,
#             is_diagonal=IS_DIAGONAL).cpu()

# if INIT_BY_SAMPLES:
#     D.init_r_by_samples(Y_sampler.sample(N_POTENTIALS))
    
# D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

In [12]:
comet_ml.login()

In [13]:
exp = comet_ml.start(
    project_name="EgNOT",
    experiment_config=comet_ml.ExperimentConfig(
        name=EXP_NAME,
        parse_args=False
    ),
)

# main hyperameters
exp.log_parameters(
    dict(
        INPUT_DATA = INPUT_DATA,
        TARGET_DATA = TARGET_DATA,
        EPSILON = EPSILON,
        LANGEVIN_SAMPLING_NOISE = CONFIG.LANGEVIN_SAMPLING_NOISE,
        ENERGY_SAMPLING_ITERATIONS = CONFIG.ENERGY_SAMPLING_ITERATIONS,
    )
)

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/nsurname17/egnot/59a14e1b67a147309cf740de49a389ec



## ALAE

In [14]:
# To download the required model run, run training_artifacts/download_all.py in the ALAE folder.

model = load_model("../ALAE/configs/ffhq.yaml", training_artifacts_dir="../ALAE/training_artifacts/ffhq/")

Plotting results during training

In [15]:
torch.manual_seed(OUTPUT_SEED); np.random.seed(OUTPUT_SEED)

inds_to_map = np.random.choice(np.arange((x_inds_test < 300).sum()), size=10, replace=False)
number_of_samples = 3

mapped_all = []
latent_to_map = torch.tensor(test_latents[x_inds_test[inds_to_map]])

inp_images = test_inp_images[x_inds_test[inds_to_map]]

init_noises = [init_noise_distribution.sample((latent_to_map.size(0),)) for _ in range(number_of_samples)]

## Training

In [16]:
# wandb.init(name=EXP_NAME, config=config)
LOSS_KEY = 'loss'

for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):
    optimizer.zero_grad()
    
    X0, X1 = X_sampler.sample(BATCH_SIZE), Y_sampler.sample(BATCH_SIZE)
    
    losses = egeot_model.loss([X0, X1])
    losses[LOSS_KEY].backward()
    
    _grad_norms = torch.nn.utils.clip_grad_norm_(egeot_model.potential.parameters(), max_norm=CONFIG.CLIP_GRAD_NORM)
    optimizer.step()
    
    exp.log_metrics({k: v.item() for k, v in losses.items()}, step=step)
    
    if step % 1000 == 0:
        mapped_all = []
        
        with torch.no_grad():
            for k in range(number_of_samples):
                mapped = egeot_model.sample(
                    latent_to_map.to(DEVICE), 
                    CONFIG.TEST_ENERGY_SAMPLING_ITERATIONS, 
                    y_init=init_noises[k]).cpu()
                mapped_all.append(mapped)

        mapped = torch.stack(mapped_all, dim=1)

        decoded_all = []
        with torch.no_grad():
            for k in range(number_of_samples):
                decoded_img = decode(model, mapped[:, k])
                decoded_img = ((decoded_img * 0.5 + 0.5) * 255).type(torch.long).clamp(0, 255).cpu().type(torch.uint8).permute(0, 2, 3, 1).numpy()
                decoded_all.append(decoded_img)

        decoded_all = np.stack(decoded_all, axis=1)
        
        fig, axes = plt.subplots(10, number_of_samples+1, figsize=(number_of_samples+1, 10), dpi=200)

        for i, ind in enumerate(range(10)):
            ax = axes[i]
            ax[0].imshow(inp_images[ind])
            for k in range(number_of_samples):
                ax[k+1].imshow(decoded_all[ind, k])

                ax[k+1].get_xaxis().set_visible(False)
                ax[k+1].set_yticks([])

            ax[0].get_xaxis().set_visible(False)
            ax[0].set_yticks([])

        fig.tight_layout(pad=0.05)
        plt.savefig(os.path.join(OUTPUT_PATH, f'mapping_{step}.png'))
        exp.log_image(image_data=os.path.join(OUTPUT_PATH, f'mapping_{step}.png'), step=step, name='mapping')
        plt.close(fig)
         
egeot_model.store(os.path.join(OUTPUT_PATH, f'potential.pt'))
torch.save(optimizer.state_dict(), os.path.join(OUTPUT_PATH, f'optimizer.pt'))

exp.end()

  0%|                                                                                                                                           | 0/10000 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/torch/autograd/graph.py:744: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [23:22<00:00,  7.13it/s]
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:    

# Results plotting

In [23]:
mapped_all = []

with torch.no_grad():
    for k in range(number_of_samples):
        y_init = init_noise_distribution.sample((latent_to_map.size(0),))
        mapped = egeot_model.sample(
            latent_to_map.to(DEVICE), 
            CONFIG.TEST_ENERGY_SAMPLING_ITERATIONS, 
            y_init=init_noises[k]
        ).cpu()
        mapped_all.append(mapped)
    
mapped = torch.stack(mapped_all, dim=1)

decoded_all = []
with torch.no_grad():
    for k in range(number_of_samples):
        decoded_img = decode(model, mapped[:, k])
        decoded_img = ((decoded_img * 0.5 + 0.5) * 255).type(torch.long).clamp(0, 255).cpu().type(torch.uint8).permute(0, 2, 3, 1).numpy()
        decoded_all.append(decoded_img)
        
decoded_all = np.stack(decoded_all, axis=1)

In [25]:
# %matplotlib inline
fig, axes = plt.subplots(10, number_of_samples+1, figsize=(number_of_samples+1, 10), dpi=200)

for i, ind in enumerate(range(10)):
    ax = axes[i]
    ax[0].imshow(inp_images[ind])
    for k in range(number_of_samples):
        ax[k+1].imshow(decoded_all[ind, k])
        
        ax[k+1].get_xaxis().set_visible(False)
        ax[k+1].set_yticks([])
        
    ax[0].get_xaxis().set_visible(False)
    ax[0].set_yticks([])

fig.tight_layout(pad=0.05)
plt.savefig(os.path.join(OUTPUT_PATH, f'mapping.png'))
plt.close(fig)

In [26]:
torch.manual_seed(OUTPUT_SEED); np.random.seed(OUTPUT_SEED)
y_init_all = init_noise_distribution.sample((X_test.size(0),))

test_batch_size = 128
for n_test_sample_steps in [1000, 2000, 3000]:
    print(f'sampling for {n_test_sample_steps}')
    mapped_all = []

    for i in tqdm(range(X_test.size(0) // test_batch_size + 1)):
        if i * test_batch_size >= X_test.size(0):
            continue
        latents_to_map = X_test[i * test_batch_size: (i + 1) * test_batch_size]
        y_init = y_init_all[i * test_batch_size: (i + 1) * test_batch_size]
        assert len(latents_to_map) > 0

        with torch.no_grad():
            mapped = egeot_model.sample(latents_to_map.to(DEVICE), n_test_sample_steps, y_init=y_init).cpu()
            mapped_all.append(mapped)

    mapped = torch.concatenate(mapped_all, dim=0)
    torch.save(mapped.cpu(), os.path.join(OUTPUT_PATH, f'X_test_mapped_{n_test_sample_steps}.pt'))

sampling for 1000


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:23<00:00,  1.48it/s]


sampling for 2000


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [00:45<00:00,  1.35s/it]


sampling for 3000


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 34/34 [01:08<00:00,  2.03s/it]
